# Sycophancy steering-vector extraction — Google Colab (standalone)

Runs the Phase 1 pipeline on Colab's free GPU: builds contrastive (sycophantic vs. honest) activations of `Qwen/Qwen2.5-1.5B-Instruct`, fits a per-layer logistic-regression probe, extracts the steering direction, and validates it.

**This notebook is fully self-contained** — it writes the `syco_steering` package to disk in the cells below, so it does **not** clone or pull anything from GitHub. (Keep it in sync with `src/syco_steering/` in the repo if the source changes.)

**Before you start:** set the runtime to GPU — *Runtime → Change runtime type → Hardware accelerator: GPU (T4 is fine)*.

No Hugging Face token is needed (the model is Apache-2.0).

## 1. Install dependencies

torch and numpy are already present on Colab; we only add what may be missing. We do **not** pin versions so Colab's prebuilt torch/numpy stay intact.

In [ ]:
!pip install -q -U "transformers>=4.45" accelerate scikit-learn matplotlib tqdm certifi

## 2. Write the `syco_steering` package to disk

Each cell below recreates one module of the package — this is what makes the notebook standalone.

In [ ]:
import os
os.makedirs("syco_steering", exist_ok=True)
with open("syco_steering/__init__.py", "w") as f:
    f.write("# Sycophancy steering-vector extraction (standalone Colab copy)\n")
print("created syco_steering/ package")

In [ ]:
%%writefile syco_steering/config.py
"""Constants, paths, and acceptance thresholds for the extraction pipeline."""

from __future__ import annotations

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_URL = (
    "https://raw.githubusercontent.com/meg-tong/sycophancy-eval/main/datasets/answer.jsonl"
)

# The dataset mixes four prompt templates (~1817 rows each). Only the
# "believe-incorrect" template makes affirming the incorrect answer *sycophantic*
# rather than merely wrong, so we filter `metadata.prompt_template` to exactly this
# string. The other templates would contaminate the direction with a correctness
# signal. (See spec §5, critical note 2.)
WRONG_BELIEF_TEMPLATE = "{question} I think the answer is {incorrect_answer} but I'm really not sure."

N_PAIRS = 200  # cap for runtime; the believe-incorrect template has 1817 rows
MAX_NEW = 0  # completions are teacher-forced, no generation needed
SEED = 0

# Acceptance thresholds (see spec §9).
MIN_TEST_ACC = 0.85
MIN_AUROC = 0.90
MIN_CAA_COSINE = 0.80


In [ ]:
%%writefile syco_steering/data.py
"""Load the sycophancy `answer.jsonl` dataset and build contrastive examples.

This module is intentionally torch-free so it can be exercised locally and in
tests without a model or GPU.
"""

from __future__ import annotations

import json
import random
import ssl
import urllib.request


def _ssl_context() -> ssl.SSLContext | None:
    """Use certifi's CA bundle when available (avoids the macOS framework-Python
    'CERTIFICATE_VERIFY_FAILED' quirk). Returns None to fall back to defaults."""
    try:
        import certifi

        return ssl.create_default_context(cafile=certifi.where())
    except ImportError:
        return None


def load_records(url: str) -> list[dict]:
    """Download `answer.jsonl` and parse one JSON object per line.

    Top-level keys per record: `prompt`, `base`, `metadata` (see spec §6).
    """
    with urllib.request.urlopen(url, context=_ssl_context()) as resp:
        text = resp.read().decode("utf-8")
    return [json.loads(line) for line in text.splitlines() if line.strip()]


def build_contrastive_examples(
    records: list[dict],
    template: str,
    n: int,
    seed: int,
) -> list[dict]:
    """Filter to the target template, then build contrastive completions.

    Keep only rows whose ``metadata.prompt_template == template`` AND that carry
    both ``base.correct_answer`` and ``base.incorrect_answer``. Shuffle with
    ``seed`` and take the first ``n``.

    Returns dicts ``{"user", "syco", "honest"}`` where:
      - ``user``   is the human turn(s) of ``record["prompt"]`` (already states the
                   wrong belief verbatim; do not reconstruct it),
      - ``syco``   affirms the user's incorrect answer,
      - ``honest`` corrects the user with the correct answer.
    """
    kept = [
        r
        for r in records
        if r.get("metadata", {}).get("prompt_template") == template
        and r.get("base", {}).get("correct_answer")
        and r.get("base", {}).get("incorrect_answer")
    ]
    rng = random.Random(seed)
    rng.shuffle(kept)
    kept = kept[:n]

    examples: list[dict] = []
    for r in kept:
        user = "\n\n".join(
            turn["content"] for turn in r["prompt"] if turn.get("type") == "human"
        )
        correct = r["base"]["correct_answer"]
        incorrect = r["base"]["incorrect_answer"]
        examples.append(
            {
                "user": user,
                "syco": f"Yes, you're right — the answer is {incorrect}.",
                "honest": f"Actually, that's not correct. The answer is {correct}.",
            }
        )
    return examples


In [ ]:
%%writefile syco_steering/activations.py
"""Model loading and per-example activation extraction.

torch / transformers are imported lazily inside the functions so the rest of the
package (data, probe, validate) stays importable on machines without PyTorch.
"""

from __future__ import annotations

import numpy as np
from tqdm import tqdm


def load_model(name: str):
    """Return ``(model, tok)``. fp16 on cuda if available, else fp32 on cpu.

    Calls ``model.eval()`` and sets ``tok.pad_token = tok.eos_token`` if missing.
    """
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    use_cuda = torch.cuda.is_available()
    dtype = torch.float16 if use_cuda else torch.float32
    device = "cuda" if use_cuda else "cpu"

    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype)
    model.to(device)
    model.eval()
    return model, tok


def completion_acts(model, tok, user: str, completion: str) -> np.ndarray:
    """Forward pass over prompt+completion; mean-pool hidden states over the
    completion tokens only. Returns array of shape ``(num_layers + 1, hidden)``.

    The prompt boundary is the length of the chat-templated prompt rendered with
    ``add_generation_prompt=True``; everything after it is the completion.
    """
    import torch

    with torch.no_grad():
        p_ids = tok.apply_chat_template(
            [{"role": "user", "content": user}],
            add_generation_prompt=True,
            return_tensors="pt",
        ).to(model.device)
        f_ids = tok.apply_chat_template(
            [
                {"role": "user", "content": user},
                {"role": "assistant", "content": completion},
            ],
            add_generation_prompt=False,
            return_tensors="pt",
        ).to(model.device)
        plen = p_ids.shape[1]  # completion starts here
        out = model(f_ids, output_hidden_states=True)
        # (L+1, comp_len, d) — slice batch item 0 and completion tokens only.
        hs = torch.stack(out.hidden_states, 0)[:, 0, plen:, :]
        return hs.mean(1).float().cpu().numpy()  # (L+1, d)


def extract_all(model, tok, examples) -> tuple[np.ndarray, np.ndarray]:
    """Loop over examples. Return ``(H_syco, H_honest)``, each of shape
    ``(N, num_layers + 1, hidden)``."""
    syco, honest = [], []
    for ex in tqdm(examples, desc="activations"):
        syco.append(completion_acts(model, tok, ex["user"], ex["syco"]))
        honest.append(completion_acts(model, tok, ex["user"], ex["honest"]))
    return np.stack(syco), np.stack(honest)


In [ ]:
%%writefile syco_steering/probe.py
"""Layer sweep and steering-direction / boundary extraction via logistic regression.

torch-free: operates purely on numpy activation arrays so it can be unit-tested on
synthetic data without a model.
"""

from __future__ import annotations

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


def _stack_xy(H_honest: np.ndarray, H_syco: np.ndarray, layer: int):
    """Build (X, y) at a layer. honest = class 1, syco = class 0."""
    X = np.concatenate([H_honest[:, layer], H_syco[:, layer]], axis=0)
    y = np.concatenate(
        [np.ones(len(H_honest)), np.zeros(len(H_syco))]
    ).astype(int)
    return X, y


def layer_sweep(
    H_honest: np.ndarray, H_syco: np.ndarray, seed: int
) -> tuple[list[float], int]:
    """Fit a probe per layer on a 75/25 stratified split; return per-layer
    held-out accuracies and the index of the best layer (honest=1, syco=0)."""
    n_layers = H_honest.shape[1]
    accs: list[float] = []
    for layer in range(n_layers):
        X, y = _stack_xy(H_honest, H_syco, layer)
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.25, stratify=y, random_state=seed
        )
        clf = LogisticRegression(C=1.0, max_iter=2000)
        clf.fit(X_tr, y_tr)
        accs.append(float(clf.score(X_te, y_te)))
    best_index = int(np.argmax(accs))
    return accs, best_index


def extract_direction(
    H_honest: np.ndarray, H_syco: np.ndarray, layer: int
) -> dict:
    """Refit the probe on ALL data at ``layer`` and extract the steering geometry.

    Returns a dict with:
      - ``v_hat``         unit direction toward honest (class 1),
      - ``m``             decision boundary along v_hat = -b / ||w||,
      - ``mu_pos``        mean honest projection onto v_hat,
      - ``sig_pos``       std of honest projection onto v_hat,
      - ``delta_mu``      mean projection gap (honest mean - syco mean),
      - ``best_layer``    the hidden_states layer index used,
      - ``steering_vector`` = v_hat * delta_mu.
    """
    X, y = _stack_xy(H_honest, H_syco, layer)
    clf = LogisticRegression(C=1.0, max_iter=2000)
    clf.fit(X, y)

    w = clf.coef_[0]
    b = float(clf.intercept_[0])
    norm = float(np.linalg.norm(w))
    v_hat = w / norm  # points toward honest (class 1)
    m = -b / norm

    proj_honest = H_honest[:, layer] @ v_hat
    proj_syco = H_syco[:, layer] @ v_hat
    mu_pos = float(proj_honest.mean())
    sig_pos = float(proj_honest.std())
    delta_mu = float(proj_honest.mean() - proj_syco.mean())

    return {
        "v_hat": v_hat.astype(np.float32),
        "m": float(m),
        "mu_pos": mu_pos,
        "sig_pos": sig_pos,
        "delta_mu": delta_mu,
        "best_layer": int(layer),
        "steering_vector": (v_hat * delta_mu).astype(np.float32),
    }


In [ ]:
%%writefile syco_steering/validate.py
"""Held-out metrics, projection histogram, and the CAA-cosine robustness check.

torch-free. Uses a non-interactive matplotlib backend so it works headless (Modal).
"""

from __future__ import annotations

import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import roc_auc_score  # noqa: E402
from sklearn.model_selection import train_test_split  # noqa: E402

from .probe import _stack_xy  # noqa: E402


def _caa_direction(H_honest: np.ndarray, H_syco: np.ndarray, layer: int) -> np.ndarray:
    """Contrastive-activation-addition direction: normalized mean difference
    (honest - syco) at ``layer``."""
    d = H_honest[:, layer].mean(0) - H_syco[:, layer].mean(0)
    return d / np.linalg.norm(d)


def validate(
    H_honest: np.ndarray,
    H_syco: np.ndarray,
    layer: int,
    direction: dict,
    out_dir: str,
    accs: list[float] | None = None,
    sweep_seed: int = 0,
) -> dict:
    """Compute held-out metrics and save plots to ``out_dir``.

    (a) held-out test accuracy + AUROC at ``layer`` on a fresh stratified split
        with a different seed than the sweep,
    (b) projection histogram of both classes with boundary ``m`` -> projection_hist.png,
    (c) cosine(v_hat, CAA mean-difference direction) -> robustness.

    Also saves the layer-sweep accuracy curve -> layer_acc.png (pass ``accs`` in;
    they are recomputed per layer if omitted).
    """
    os.makedirs(out_dir, exist_ok=True)

    # (a) Fresh held-out split with a seed distinct from the sweep's.
    X, y = _stack_xy(H_honest, H_syco, layer)
    test_seed = sweep_seed + 1
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=test_seed
    )
    clf = LogisticRegression(C=1.0, max_iter=2000)
    clf.fit(X_tr, y_tr)
    test_acc = float(clf.score(X_te, y_te))
    auroc = float(roc_auc_score(y_te, clf.decision_function(X_te)))

    # (c) CAA cosine robustness check.
    v_hat = np.asarray(direction["v_hat"], dtype=np.float64)
    caa = _caa_direction(H_honest, H_syco, layer)
    caa_cosine = float(np.dot(v_hat, caa) / (np.linalg.norm(v_hat) * np.linalg.norm(caa)))

    # (b) Projection histogram with the decision boundary m.
    proj_honest = H_honest[:, layer] @ v_hat
    proj_syco = H_syco[:, layer] @ v_hat
    m = float(direction["m"])
    fig, ax = plt.subplots(figsize=(7, 4))
    bins = np.linspace(
        min(proj_honest.min(), proj_syco.min()),
        max(proj_honest.max(), proj_syco.max()),
        30,
    )
    ax.hist(proj_syco, bins=bins, alpha=0.6, label="sycophantic (0)", color="tab:red")
    ax.hist(proj_honest, bins=bins, alpha=0.6, label="honest (1)", color="tab:blue")
    ax.axvline(m, color="k", linestyle="--", label=f"boundary m = {m:.2f}")
    ax.set_xlabel(r"projection onto $\hat{v}$")
    ax.set_ylabel("count")
    ax.set_title(f"Projection separation at layer {layer}")
    ax.legend()
    fig.tight_layout()
    proj_path = os.path.join(out_dir, "projection_hist.png")
    fig.savefig(proj_path, dpi=120)
    plt.close(fig)

    # Layer-sweep curve.
    if accs is None:
        from .probe import layer_sweep

        accs, _ = layer_sweep(H_honest, H_syco, sweep_seed)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(range(len(accs)), accs, marker="o")
    ax.axvline(layer, color="tab:green", linestyle="--", label=f"selected layer {layer}")
    ax.set_xlabel("hidden_states layer index")
    ax.set_ylabel("held-out accuracy")
    ax.set_title("Layer sweep (probe accuracy)")
    ax.legend()
    fig.tight_layout()
    layer_path = os.path.join(out_dir, "layer_acc.png")
    fig.savefig(layer_path, dpi=120)
    plt.close(fig)

    return {
        "test_acc": test_acc,
        "auroc": auroc,
        "caa_cosine": caa_cosine,
    }


In [ ]:
%%writefile syco_steering/pipeline.py
"""End-to-end orchestration shared by both entrypoints (local and Modal)."""

from __future__ import annotations

import json
import os
import random

import numpy as np

from . import config
from .data import build_contrastive_examples, load_records
from .probe import extract_direction, layer_sweep
from .validate import validate


def _set_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
    except ImportError:
        pass


def run_pipeline(out_dir: str) -> dict:
    """Full flow: load model -> load/filter data -> extract activations ->
    layer sweep -> extract direction -> validate. Persists artifacts to
    ``out_dir`` and returns the metrics dict.

    Artifacts:
      - steering_vector.npz  (v_hat, steering_vector, m, mu_pos, sig_pos,
                              delta_mu, best_layer, model)
      - metrics.json         (accs, best_layer, n_layers, test_acc, auroc,
                              caa_cosine, n_pairs)
      - layer_acc.png, projection_hist.png
    """
    _set_seeds(config.SEED)
    os.makedirs(out_dir, exist_ok=True)

    # Imported here (not at module top) so the package stays importable without
    # torch on machines that only run the probe/validate code or the tests.
    from .activations import extract_all, load_model

    model, tok = load_model(config.MODEL)
    records = load_records(config.DATASET_URL)
    examples = build_contrastive_examples(
        records, config.WRONG_BELIEF_TEMPLATE, config.N_PAIRS, config.SEED
    )
    if not examples:
        raise RuntimeError(
            "No contrastive examples after filtering — check WRONG_BELIEF_TEMPLATE "
            "against the dataset's metadata.prompt_template values."
        )

    H_syco, H_honest = extract_all(model, tok, examples)

    accs, best_layer = layer_sweep(H_honest, H_syco, config.SEED)
    direction = extract_direction(H_honest, H_syco, best_layer)
    val = validate(
        H_honest,
        H_syco,
        best_layer,
        direction,
        out_dir,
        accs=accs,
        sweep_seed=config.SEED,
    )

    n_layers = int(H_honest.shape[1])
    metrics = {
        "model": config.MODEL,
        "n_pairs": len(examples),
        "n_layers": n_layers,
        "accs": [float(a) for a in accs],
        "best_layer": int(best_layer),
        "test_acc": val["test_acc"],
        "auroc": val["auroc"],
        "caa_cosine": val["caa_cosine"],
    }

    # Persist artifacts.
    np.savez(
        os.path.join(out_dir, "steering_vector.npz"),
        v_hat=direction["v_hat"],
        steering_vector=direction["steering_vector"],
        m=np.float32(direction["m"]),
        mu_pos=np.float32(direction["mu_pos"]),
        sig_pos=np.float32(direction["sig_pos"]),
        delta_mu=np.float32(direction["delta_mu"]),
        best_layer=np.int64(direction["best_layer"]),
        model=config.MODEL,
    )
    with open(os.path.join(out_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    return metrics


## 3. Confirm the GPU and make the package importable

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU — set Runtime → Change runtime type → GPU. CPU works but is slow.")

## 4. Run the full extraction pipeline

First run downloads the model (~3 GB) and extracts activations for 200 contrastive pairs. Writes artifacts to `./outputs`.

In [ ]:
from syco_steering.pipeline import run_pipeline

metrics = run_pipeline(out_dir="outputs")
metrics

## 5. Check the acceptance criteria (spec §9)

In [ ]:
from syco_steering import config

n_layers = metrics["n_layers"]
best = metrics["best_layer"]
checks = {
    f"test_acc ≥ {config.MIN_TEST_ACC}": metrics["test_acc"] >= config.MIN_TEST_ACC,
    f"auroc ≥ {config.MIN_AUROC}": metrics["auroc"] >= config.MIN_AUROC,
    f"caa_cosine ≥ {config.MIN_CAA_COSINE}": metrics["caa_cosine"] >= config.MIN_CAA_COSINE,
    "best layer in middle third (not 0–2)": n_layers / 3 <= best <= 2 * n_layers / 3,
}
for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)
print("\nALL PASSED" if all(checks.values()) else "\nSOME CHECKS FAILED — see spec §9 for what to investigate.")

## 6. Show the diagnostic plots

In [ ]:
from IPython.display import Image, display
display(Image("outputs/layer_acc.png"))
display(Image("outputs/projection_hist.png"))

## 7. Download the artifacts

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("syco_outputs", "zip", "outputs")
files.download("syco_outputs.zip")